In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/label_mapping.json
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0079.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0018.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0032.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0049.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0215.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0064.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0016.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0114.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0090.jpg

In [4]:
# ==============================================================================
# KAGGLE NOTEBOOK: RICE LEAF DISEASE CLASSIFICATION USING LOGISTIC REGRESSION
# ==============================================================================
# Metric: Macro F1-Score | Kỹ thuật: HSV Color Histogram + Gray Pixels + Logistic Regression
# Author: Gemini Data Science Assistant
# ==============================================================================

import os
import json
import cv2
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings("ignore")

# ------------------------------------------------------------------------------
# 1. CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ (Sử dụng đường dẫn cũ bạn đã chạy thành công)
# ------------------------------------------------------------------------------
DATA_DIR = "/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup" # <-- Hãy sửa lại tên thư mục chính xác của bạn tại đây
TRAIN_DIR = os.path.join(DATA_DIR, "/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/train")
TEST_DIR = os.path.join(DATA_DIR, "/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup")
MAPPING_FILE = os.path.join(DATA_DIR, "/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/label_mapping.json")

# Kích thước ảnh resize
IMG_SIZE = (64, 64) 

# Đọc file label mapping
with open(MAPPING_FILE, "r") as f:
    label_mapping = json.load(f)
print("--- Bản đồ ánh xạ nhãn (Label Mapping) ---")
print(json.dumps(label_mapping, indent=4))


# ------------------------------------------------------------------------------
# 2. HÀM TRÍCH XUẤT ĐẶC TRƯNG HÌNH ẢNH TRUYỀN THỐNG (FEATURE EXTRACTION)
# ------------------------------------------------------------------------------
def extract_features(img_path, img_size=(64, 64)):
    """
    Trích xuất đặc trưng kết hợp:
    1. HSV Color Histogram: Đại diện cho màu sắc bệnh (đốm nâu, cháy lá, vàng lá).
    2. Raw Pixel Grayscale: Giữ lại hình thái cấu trúc đốm bệnh và gân lá.
    """
    img = cv2.imread(img_path)
    if img is None:
        return None
    
    # Chuyển đổi không gian màu sang RGB và HSV
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    img_resized_rgb = cv2.resize(img_rgb, img_size)
    img_resized_gray = cv2.cvtColor(img_resized_rgb, cv2.COLOR_RGB2GRAY)
    
    # 1. Trích xuất lược đồ màu sắc (Kênh màu H và S)
    hist_h = cv2.calcHist([img_hsv], [0], None, [16], [0, 180])
    hist_s = cv2.calcHist([img_hsv], [1], None, [16], [0, 256])
    
    # Chuẩn hóa Histogram
    cv2.normalize(hist_h, hist_h)
    cv2.normalize(hist_s, hist_s)
    color_feat = np.concatenate([hist_h.flatten(), hist_s.flatten()])
    
    # 2. Đặc trưng pixel thô từ ảnh xám (Chuẩn hóa về [0, 1])
    gray_feat = img_resized_gray.flatten() / 255.0
    
    # Ghép 2 loại vector đặc trưng
    features = np.concatenate([color_feat, gray_feat])
    return features


# ------------------------------------------------------------------------------
# 3. TẢI TẬP DỮ LIỆU HUẤN LUYỆN (TRAIN DATASET)
# ------------------------------------------------------------------------------
print("\n--- Đang tải dữ liệu huấn luyện (Train Data) ---")
X_train_list = []
y_train_list = []

for class_name, label_idx in label_mapping.items():
    class_folder = os.path.join(TRAIN_DIR, class_name)
    if not os.path.exists(class_folder):
        print(f"⚠️ Thư mục '{class_name}' không tồn tại, bỏ qua.")
        continue
        
    print(f"Đang tải lớp: {class_name} -> Nhãn: {label_idx}")
    img_names = os.listdir(class_folder)
    for img_name in img_names:
        if img_name.startswith('.'): 
            continue
        img_path = os.path.join(class_folder, img_name)
        feat = extract_features(img_path, img_size=IMG_SIZE)
        if feat is not None:
            X_train_list.append(feat)
            y_train_list.append(label_idx)

X_train = np.array(X_train_list)
y_train = np.array(y_train_list)
print(f"-> Hoàn thành tải Train! Kích thước mảng: X_train = {X_train.shape}, y_train = {y_train.shape}")


# ------------------------------------------------------------------------------
# 4. ĐÁNH GIÁ MÔ HÌNH BẰNG STRATIFIED K-FOLD
# ------------------------------------------------------------------------------
print("\n--- Bắt đầu kiểm thử chéo (Stratified 5-Fold Cross Validation) ---")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    
    # Chuẩn hóa đặc trưng (Z-score Scaling rất quan trọng với Logistic Regression)
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    
    # Khởi tạo mô hình Logistic Regression
    # max_iter=1000 giúp thuật toán có đủ số vòng lặp để hội tụ
    # class_weight='balanced' tối ưu hóa việc phân phối trọng số cho các lớp để đẩy cao F1-Macro
    lr_model = LogisticRegression(
        solver='lbfgs', 
        multi_class='multinomial', 
        C=1.0, 
        max_iter=1000, 
        class_weight='balanced', 
        random_state=42
    )
    lr_model.fit(X_tr_scaled, y_tr)
    
    # Dự đoán trên tập validation độc lập
    y_pred = lr_model.predict(X_val_scaled)
    score = f1_score(y_val, y_pred, average="macro")
    fold_scores.append(score)
    print(f"Fold {fold+1} - Macro F1-Score: {score:.4f}")

print(f">> Điểm F1-Macro trung bình của Logistic Regression: {np.mean(fold_scores):.4f}")


# ------------------------------------------------------------------------------
# 5. HUẤN LUYỆN TOÀN BỘ DỮ LIỆU TRÊN MODEL CUỐI CÙNG
# ------------------------------------------------------------------------------
print("\n--- Huấn luyện mô hình Logistic Regression trên toàn bộ dữ liệu mẫu ---")
final_scaler = StandardScaler()
X_train_scaled = final_scaler.fit_transform(X_train)

final_lr = LogisticRegression(
    solver='lbfgs', 
    multi_class='multinomial', 
    C=1.0, 
    max_iter=1000, 
    class_weight='balanced', 
    random_state=42
)
final_lr.fit(X_train_scaled, y_train)


# ------------------------------------------------------------------------------
# 6. DỰ ĐOÁN TRÊN TẬP TEST VÀ TẠO FILE SUBMISSION.CSV CHUẨN KAGGLE
# ------------------------------------------------------------------------------
print("\n--- Đang trích xuất đặc trưng tập kiểm thử Test & Suy luận ---")
test_images = sorted([f for f in os.listdir(TEST_DIR) if not f.startswith('.')])
test_features = []
valid_image_names = []

for img_name in test_images:
    img_path = os.path.join(TEST_DIR, img_name)
    feat = extract_features(img_path, img_size=IMG_SIZE)
    if feat is not None:
        test_features.append(feat)
        valid_image_names.append(img_name)

X_test = np.array(test_features)
X_test_scaled = final_scaler.transform(X_test)

# Tiến hành dự đoán nhãn bằng Logistic Regression
test_predictions = final_lr.predict(X_test_scaled)

# Tạo DataFrame theo đúng chuẩn định dạng nộp bài của Kaggle
# Lưu ý: Chỉnh lại tên cột 'image_id' và 'label' nếu file mẫu sample_submission.csv dùng từ khác
submission_df = pd.DataFrame({
    "image_id": valid_image_names,
    "label": test_predictions
})

# Xuất file bài nộp submission.csv
submission_df.to_csv("submission.csv", index=False)
print("\n 🎉 QUÁ TRÌNH HOÀN THÀNH XUẤT SẮC!")
print("File 'submission.csv' (Logistic Regression) đã tạo thành công.")
print(submission_df.head(10))

--- Bản đồ ánh xạ nhãn (Label Mapping) ---
{
    "Bacterial Leaf Blight": 0,
    "Brown Spot": 1,
    "Healthy Rice Leaf": 2,
    "Leaf Blast": 3,
    "Leaf scald": 4,
    "Narrow Brown Leaf Spot": 5,
    "Rice Hispa": 6,
    "Sheath Blight": 7
}

--- Đang tải dữ liệu huấn luyện (Train Data) ---
Đang tải lớp: Bacterial Leaf Blight -> Nhãn: 0
Đang tải lớp: Brown Spot -> Nhãn: 1
Đang tải lớp: Healthy Rice Leaf -> Nhãn: 2
Đang tải lớp: Leaf Blast -> Nhãn: 3
Đang tải lớp: Leaf scald -> Nhãn: 4
Đang tải lớp: Narrow Brown Leaf Spot -> Nhãn: 5
Đang tải lớp: Rice Hispa -> Nhãn: 6
Đang tải lớp: Sheath Blight -> Nhãn: 7
-> Hoàn thành tải Train! Kích thước mảng: X_train = (2026, 4128), y_train = (2026,)

--- Bắt đầu kiểm thử chéo (Stratified 5-Fold Cross Validation) ---
Fold 1 - Macro F1-Score: 0.7100
Fold 2 - Macro F1-Score: 0.6670
Fold 3 - Macro F1-Score: 0.6930
Fold 4 - Macro F1-Score: 0.6622
Fold 5 - Macro F1-Score: 0.6797
>> Điểm F1-Macro trung bình của Logistic Regression: 0.6824

--- Huấn 